## load data and inspect

In [ ]:
import pandas as pd

In [ ]:
df_snip = pd.read_parquet(r'C:\Users\[REDACTED-USER]\OneDrive - [REDACTED-EMPLOYER]\repos\UCLAI\data\objective-TeamA-2020\2020\2020-06\2020-06-01\2020-06-01-TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93.parquet')

In [ ]:
df_snip.columns

In [ ]:
df_snip.head(10)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import pandas as pd
import numpy as np
from IPython.display import HTML
from matplotlib.cm import get_cmap
from matplotlib.colors import Normalize

# Assuming df_snip is already loaded and sorted by 'time'
df = df_snip.copy()
df['time'] = pd.to_datetime(df['time'])  # if not already
df = df.sort_values('time').reset_index(drop=True)

# Set up normalization and colormap for heart rate
norm = Normalize(vmin=df['heart_rate'].min(), vmax=df['heart_rate'].max())
cmap = get_cmap('plasma')  # or 'viridis', 'inferno', etc.

# Set up the plot
fig, ax = plt.subplots(figsize=(8, 6))

# Set up a ScalarMappable for the colorbar
from matplotlib.cm import ScalarMappable

sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # Dummy data for colorbar

# Add colorbar to the plot
cbar = plt.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label('Heart Rate', color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.get_yticklabels(), color='white')



sc = ax.scatter([], [], s=100)  # Main point
trail, = ax.plot([], [], lw=1, color='gray', alpha=0.5)  # Optional trail

# Axis limits
ax.set_xlim(df['lon'].min() - 0.001, df['lon'].max() + 0.001)
ax.set_ylim(df['lat'].min() - 0.001, df['lat'].max() + 0.001)
ax.set_title("Object Movement Colored by Heart Rate")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

# Optional: background color and aesthetics
ax.set_facecolor('black')
fig.patch.set_facecolor('black')
ax.tick_params(colors='white')
ax.spines[:].set_color('white')

# Initialize function
def init():
    sc.set_offsets(np.empty((0, 2)))
    trail.set_data([], [])
    return sc, trail

# Animation update function
def update(frame):
    row = df.iloc[frame]
    x, y = row['lon'], row['lat']
    heart_rate = row['heart_rate']
    color = cmap(norm(heart_rate))

    sc.set_offsets([[x, y]])
    sc.set_color(color)
    
    # Trail effect
    trail.set_data(df['lon'].iloc[:frame+1], df['lat'].iloc[:frame+1])

    
    # Update the title with timestamp
    current_time = df['time'].iloc[frame]
    ax.set_title(f"Nani Movement — {current_time.strftime('%H:%M:%S')}", color='white')
    
    return sc, trail

# Animate
step = 3000
frames = range(0, len(df), step)

ani = animation.FuncAnimation(
    fig, update, frames=frames, init_func=init,
    blit=True, interval=200
)


# # To save to mp4
from matplotlib.animation import PillowWriter
ani.save("movement_heart_rate2.gif", writer=PillowWriter(fps=20), dpi=200)

# from IPython.display import HTML
from IPython.display import HTML
HTML(ani.to_jshtml())

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.cm import get_cmap, ScalarMappable
from matplotlib.colors import Normalize
from IPython.display import HTML

# Step 1: Load and process data
base_path = r'C:\Users\[REDACTED-USER]\OneDrive - [REDACTED-EMPLOYER]\repos\UCLAI\data\objective-TeamA-2020\2020\2020-06\2020-06-01'
parquet_files = [os.path.join(base_path, f) for f in os.listdir(base_path) if f.endswith('.parquet')]
df_list = [pd.read_parquet(f) for f in parquet_files]
df_snip = pd.concat(df_list, ignore_index=True)

# Rename player names
custom_names = ['Onana', 'Cole', 'Terry', 'Ramos', 'Walker', 'Giggsy', 'Xhaka', 'Scholes', 'Nani', 'Ronaldo', 'Messi']
unique_players = df_snip['player_name'].unique()
name_map = {original: custom_names[i % len(custom_names)] for i, original in enumerate(unique_players)}
df_snip['player_name'] = df_snip['player_name'].map(name_map)

# Convert time and sort
df_snip['time'] = pd.to_datetime(df_snip['time'])
df_snip = df_snip.sort_values('time').reset_index(drop=True)

# Setup for animation
players = df_snip['player_name'].unique()
norm = Normalize(vmin=df_snip['heart_rate'].min(), vmax=df_snip['heart_rate'].max())
cmap = get_cmap('plasma')
unique_times = df_snip['time'].sort_values().unique()

# Set up plot
fig, ax = plt.subplots(figsize=(10, 8))

# Colorbar
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label('Heart Rate', color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.get_yticklabels(), color='white')

# Axes config
pad_x = (df_snip['lon'].max() - df_snip['lon'].min()) * 0.05
pad_y = (df_snip['lat'].max() - df_snip['lat'].min()) * 0.05
ax.set_xlim(0,120)#(df_snip['lon'].min() - pad_x, df_snip['lon'].max() + pad_x)
ax.set_ylim(0,120)#(df_snip['lat'].min() - pad_y, df_snip['lat'].max() + pad_y)
ax.set_facecolor('black')
fig.patch.set_facecolor('black')
ax.tick_params(colors='white')
ax.spines[:].set_color('white')
ax.set_xlabel("Longitude", color='white')
ax.set_ylabel("Latitude", color='white')
title = ax.set_title('', color='white')

# Initialize dots and labels
scatters = {player: ax.plot([], [], 'o', markersize=8)[0] for player in players}
labels = {player: ax.text(0, 0, '', color='white', fontsize=8, ha='center') for player in players}

def init():
    for sc in scatters.values():
        sc.set_data([], [])
    for label in labels.values():
        label.set_text('')
    return list(scatters.values()) + list(labels.values())

def update(frame_idx):
    current_time = unique_times[frame_idx]
    frame_df = df_snip[df_snip['time'] == current_time]

    for player in players:
        player_row = frame_df[frame_df['player_name'] == player]
        if not player_row.empty:
            x = player_row['lon'].values[0]
            y = player_row['lat'].values[0]
            hr = player_row['heart_rate'].values[0]
            color = cmap(norm(hr))

            scatters[player].set_data([x], [y])
            scatters[player].set_color(color)
            labels[player].set_position((x, y + pad_y * 0.2))
            labels[player].set_text(player)
        else:
            scatters[player].set_data([], [])
            labels[player].set_text('')
    
    title.set_text(f"Player Movement — {current_time.strftime('%H:%M:%S')}")
    return list(scatters.values()) + list(labels.values()) + [title]

# Animate
step = 10000  # adjust as needed for speed
frames = range(0, len(unique_times), step)
ani = animation.FuncAnimation(fig, update, frames=frames, init_func=init,
                              blit=True, interval=300)

# To display in notebook
HTML(ani.to_jshtml())

# To save
# ani.save("team_movement.gif", writer=animation.PillowWriter(fps=10), dpi=200)


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.cm import get_cmap, ScalarMappable
from matplotlib.colors import Normalize

# === Load & Prepare Data ===
base_path = r'C:\Users\[REDACTED-USER]\OneDrive - [REDACTED-EMPLOYER]\repos\UCLAI\data\objective-TeamA-2020\2020\2020-06\2020-06-01'
parquet_files = [os.path.join(base_path, f) for f in os.listdir(base_path) if f.endswith('.parquet')]
df_list = [pd.read_parquet(f) for f in parquet_files]
df_snip = pd.concat(df_list, ignore_index=True)

# Rename player names
custom_names = ['Onana', 'Cole', 'Terry', 'Ramos', 'Walker', 'Giggsy', 'Xhaka', 'Scholes', 'Nani', 'Ronaldo', 'Messi']
unique_players = df_snip['player_name'].unique()
name_map = {original: custom_names[i % len(custom_names)] for i, original in enumerate(unique_players)}
df_snip['player_name'] = df_snip['player_name'].map(name_map)

# Convert and sort time
df_snip['time'] = pd.to_datetime(df_snip['time'])
df_snip = df_snip.sort_values('time').reset_index(drop=True)

# Get all player names
players = df_snip['player_name'].unique()

# === Loop over players ===
for player in players:
    df_player = df_snip[df_snip['player_name'] == player].copy()
    df_player = df_player.sort_values('time').reset_index(drop=True)
    times = df_player['time'].unique()

    norm = Normalize(vmin=df_snip['heart_rate'].min(), vmax=df_snip['heart_rate'].max())
    cmap = get_cmap('plasma')

    fig, ax = plt.subplots(figsize=(8, 6))

    # Colorbar
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label('Heart Rate', color='white')
    cbar.ax.yaxis.set_tick_params(color='white')
    plt.setp(cbar.ax.get_yticklabels(), color='white')

    # Axes setup
    pad_x = (df_player['lon'].max() - df_player['lon'].min()) * 0.05
    pad_y = (df_player['lat'].max() - df_player['lat'].min()) * 0.05
    ax.set_xlim(df_player['lon'].min() - pad_x, df_player['lon'].max() + pad_x)
    ax.set_ylim(df_player['lat'].min() - pad_y, df_player['lat'].max() + pad_y)
    ax.set_facecolor('black')
    fig.patch.set_facecolor('black')
    ax.tick_params(colors='white')
    ax.spines[:].set_color('white')
    ax.set_xlabel("Longitude", color='white')
    ax.set_ylabel("Latitude", color='white')
    title = ax.set_title('', color='white')

    scatter, = ax.plot([], [], 'o', color='white', markersize=8)
    label = ax.text(0, 0, '', color='white', fontsize=8, ha='center')

    def init():
        scatter.set_data([], [])
        label.set_text('')
        return scatter, label

    def update(frame_idx):
        row = df_player.iloc[frame_idx]
        x = row['lon']
        y = row['lat']
        hr = row['heart_rate']
        color = cmap(norm(hr))

        scatter.set_data([x], [y])
        scatter.set_color(color)
        label.set_position((x, y + pad_y * 0.2))
        label.set_text(player)
        title.set_text(f"{player} — {row['time'].strftime('%H:%M:%S')}")

        return scatter, label, title

    step = 10000  # frame skipping
    frames = range(0, len(df_player), step)

    ani = animation.FuncAnimation(
        fig, update, frames=frames, init_func=init,
        blit=True, interval=300
    )

    out_path = f"movement_{player}.gif"
    ani.save(out_path, writer=animation.PillowWriter(fps=10), dpi=200)
    plt.close(fig)

    print(f"Saved {out_path}")
